# Agente de análise de estoque (LangGraph + Ollama)

Demonstração dos cenários do agente. O código está em [`agente.py`](../agente.py) e o funcionamento (ferramentas, regra de situação, nós do grafo) está explicado no [README](../README.md#como-funciona).

Pré-requisitos:
- API de estoque no ar em `http://127.0.0.1:8000` (repositório `api-estoque`).
- Ollama rodando com o modelo `qwen2.5:7b` (local e gratuito).
- Kernel do notebook: o Python do `.venv` deste projeto.

Cada chamada a `perguntar()` mostra a intenção, o retorno da API e a resposta (evidência). No cadastro, as respostas do humano (campos faltantes e confirmação s/n) são passadas em `respostas` para o notebook rodar sem travar; sem esse parâmetro, o agente pergunta com `input()`.

## Carregar o agente

In [1]:
import sys
sys.path.insert(0, "..")  # o código do agente fica em agente.py, na raiz do projeto

from agente import API_URL, MODELO, agente, perguntar

print(f"Configuração carregada (API: {API_URL} | modelo: {MODELO})")
print(agente.get_graph().draw_mermaid())  # diagrama do grafo em Mermaid

Configuração carregada (API: http://127.0.0.1:8000 | modelo: qwen2.5:7b)
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpretar(interpretar)
	consultar_api(consultar_api)
	responder(responder)
	responder_direto(responder_direto)
	preparar_cadastro(preparar_cadastro)
	completar_cadastro(completar_cadastro)
	confirmar_cadastro(confirmar_cadastro)
	responder_cadastro(responder_cadastro)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpretar;
	completar_cadastro -.-> confirmar_cadastro;
	completar_cadastro -.-> responder_cadastro;
	confirmar_cadastro --> responder_cadastro;
	consultar_api --> responder;
	interpretar -.-> consultar_api;
	interpretar -.-> preparar_cadastro;
	interpretar -.-> responder_direto;
	preparar_cadastro -.-> completar_cadastro;
	preparar_cadastro -.-> confirmar_cadastro;
	preparar_cadastro -.-> responder_cadastro;
	responder --> __end__;
	responder_cadastro --> __end__;
	responder_direto --> __end__;
	classD

## Produto existente (baixo, esgotado e disponível)

In [2]:
perguntar("Qual a situação do estoque do produto PRD-001?")
perguntar("Ainda tem PRD-004 em estoque?")
perguntar("Como está o PRD-002?");

Pergunta : Qual a situação do estoque do produto PRD-001?
Intenção : consultar_produto | código: PRD-001 | situação: baixo
API      : {'ok': True, 'produto': {'codigo': 'PRD-001', 'nome': 'Mouse sem fio', 'categoria': 'Periféricos', 'quantidade_atual': 5, 'estoque_minimo': 10, 'unidade': 'un'}}
Resposta : O produto PRD-001 está em situação de baixo estoque. A quantidade atual é de 5 unidades, enquanto o estoque mínimo é de 10 unidades.
Mouse sem fio (PRD-001): 5 un em estoque, estoque mínimo 10, situação: baixo.

n8n      : {'alerta': True, 'mensagem': 'Repor o produto PRD-001: situação baixo.'}

Pergunta : Ainda tem PRD-004 em estoque?
Intenção : consultar_produto | código: PRD-004 | situação: esgotado
API      : {'ok': True, 'produto': {'codigo': 'PRD-004', 'nome': 'Cabo HDMI 2 m', 'categoria': 'Cabos e Adaptadores', 'quantidade_atual': 0, 'estoque_minimo': 20, 'unidade': 'un'}}
Resposta : A quantidade atual de PRD-004 é 0 unidades, com estoque mínimo de 20 unidades. Situação: esgota

## Produto inexistente e falhas da API

In [15]:
perguntar("Qual o estoque do PRD-999?")      # 404
perguntar("Consulte o produto PRD-503")       # 503 simulado
perguntar("Quantas unidades tem o ABC-1?");   # código inválido, barrado antes da chamada

Pergunta : Qual o estoque do PRD-999?
Intenção : consultar_produto | código: PRD-999 | situação: None
API      : {'ok': False, 'erro': 'produto_nao_encontrado', 'mensagem': 'Produto não encontrado.'}
Resposta : Não consegui obter os dados do produto PRD-999: Produto não encontrado.

Pergunta : Consulte o produto PRD-503
Intenção : consultar_produto | código: PRD-503 | situação: None
API      : {'ok': False, 'erro': 'servico_indisponivel', 'mensagem': 'Serviço temporariamente indisponível. Tente novamente mais tarde.'}
Resposta : Não consegui obter os dados do produto PRD-503: Serviço temporariamente indisponível. Tente novamente mais tarde.

Pergunta : Quantas unidades tem o ABC-1?
Intenção : consultar_produto | código: ABC-1 | situação: None
API      : {'ok': False, 'erro': 'codigo_invalido', 'mensagem': "O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001)."}
Resposta : Não consegui obter os dados do produto ABC-1: O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001).



## Sem consulta à API (conceitual, ambígua e fora do escopo)

In [16]:
perguntar("O que é estoque mínimo?")
perguntar("Como está o estoque do mouse?")
perguntar("Exclua o produto PRD-001");

Pergunta : O que é estoque mínimo?
Intenção : conceitual | código: None | situação: None
API      : não consultada
Resposta : O estoque mínimo é o nível de estoque abaixo do qual o produto precisa ser reabastecido para evitar esgotamento. Quando a quantidade atual atinge ou cai abaixo desse nível, indica que é necessário realizar uma reposição para manter a disponibilidade do produto na loja.

Pergunta : Como está o estoque do mouse?
Intenção : ambigua | código: None | situação: None
API      : não consultada
Resposta : Não identifiquei qual produto você quer consultar. Informe o código no formato PRD-XXX (ex.: PRD-001).

Pergunta : Exclua o produto PRD-001
Intenção : fora_do_escopo | código: PRD-001 | situação: None
API      : não consultada
Resposta : Só posso consultar o estoque e cadastrar novos produtos (com sua confirmação). Não altero nem excluo produtos.



## Cadastro de produto (POST /produtos, com confirmação humana e pedido dos campos faltantes)

In [17]:
perguntar("Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un",
          respostas=["s"])
perguntar("Qual a situação do estoque do PRD-020?")
perguntar("Quero cadastrar o PRD-030, uma Webcam HD da categoria Periféricos, com 8 unidades",
          respostas=["nome Webcam HD, estoque mínimo 3, unidade un", "s"]);  # frase incompleta: o agente pergunta o que falta

Agente   : Confirma o cadastro de PRD-020 - Teclado mecânico (Periféricos), 15 un, estoque mínimo 5? (s/n)
Humano   : s
Pergunta : Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un
Intenção : cadastrar_produto | código: PRD-020 | situação: None
API      : {'ok': False, 'erro': 'produto_duplicado', 'mensagem': 'Já existe um produto cadastrado com o código PRD-020.'}
Resposta : Não foi possível cadastrar o produto: Já existe um produto cadastrado com o código PRD-020.

Pergunta : Qual a situação do estoque do PRD-020?
Intenção : consultar_produto | código: PRD-020 | situação: disponível
API      : {'ok': True, 'produto': {'codigo': 'PRD-020', 'nome': 'Teclado mecânico', 'categoria': 'Periféricos', 'quantidade_atual': 15, 'estoque_minimo': 5, 'unidade': 'un'}}
Resposta : O produto PRD-020 (Teclado mecânico) está disponível, com 15 unidades em estoque e estoque mínimo de 5 unidades.

Agente   : Para cadastrar, inform

## Falhas no cadastro (código duplicado e dados inválidos)

In [18]:
perguntar("Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un",
          respostas=["s"])  # Deve retornar 409: o PRD-020 já existe
perguntar("Cadastre o produto PRD-021: nome Mouse gamer, categoria Periféricos, quantidade atual -3, estoque mínimo 2, unidade un");  # barrado antes da confirmação

Agente   : Confirma o cadastro de PRD-020 - Teclado mecânico (Periféricos), 15 un, estoque mínimo 5? (s/n)
Humano   : s
Pergunta : Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un
Intenção : cadastrar_produto | código: PRD-020 | situação: None
API      : {'ok': False, 'erro': 'produto_duplicado', 'mensagem': 'Já existe um produto cadastrado com o código PRD-020.'}
Resposta : Não foi possível cadastrar o produto: Já existe um produto cadastrado com o código PRD-020.

Pergunta : Cadastre o produto PRD-021: nome Mouse gamer, categoria Periféricos, quantidade atual -3, estoque mínimo 2, unidade un
Intenção : cadastrar_produto | código: PRD-021 | situação: None
API      : {'ok': False, 'erro': 'dados_invalidos', 'mensagem': 'Dados inválidos para cadastro (quantidade_atual: Input should be greater than or equal to 0).'}
Resposta : Não foi possível cadastrar o produto: Dados inválidos para cadastro (quantidade_atual: I